# Detection Threshold

Compare face probs on real faces vs random objects. `detection_prob_threshold = 0.87`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from facenet_models import FacenetModel
from core.normalize import resize_images

%matplotlib inline

In [ ]:
root = Path.cwd().parent / "data" / "test_images"
resize_images(root, max_width=1280)

face_dirs = [root / "Team's Faces", root / "Online Faces"]
object_dir = root / "Random Objects (Noise)"
exts = {".jpg", ".jpeg", ".png"}


def load_image(path):
    img = io.imread(str(path))
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    if img.shape[-1] == 4:
        img = img[..., :3]
    if img.dtype != np.uint8:
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)
    return img


def get_probs(folder):
    probs = []
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() not in exts:
            continue
        boxes, p, _ = model.detect(load_image(path))
        if boxes is None:
            continue
        probs.extend(float(x) for x in p if x is not None)
    return np.array(probs)


model = FacenetModel()

face_probs = np.concatenate([get_probs(d) for d in face_dirs])
object_probs = get_probs(object_dir)

print("faces:", np.round(sorted(face_probs), 3))
print("objects:", np.round(sorted(object_probs), 3))
print("face min:", face_probs.min(), "object max (junk-ish):", object_probs[object_probs < 0.9].max())

In [ ]:
plt.hist(face_probs, bins=20, alpha=0.6, label="faces")
plt.hist(object_probs, bins=20, alpha=0.6, label="objects")
plt.axvline(0.87, color="black", ls="--", label="0.87")
plt.xlabel("detection probability")
plt.legend()
plt.show()

In [ ]:
detection_prob_threshold = 0.87